## Setup

In [ ]:
# Put all imports here #######################
import sys, json
from pathlib import Path
import numpy as np
import torch
from rdkit import Chem

##############################################

SRC = next((p / "src" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "lead_optimization.py").is_file()), None)
if SRC is None:
    raise RuntimeError(
        "cannot find lead_optimization.py -- run this notebook from inside the "
        "MolPLAtte repo, or set SRC to <repo>/molplatte/src by hand")
sys.path.insert(0, str(SRC))
print("src:", SRC)


from lead_optimization import LeadOptimizer, pocket_embedding_from_structure
from lead_report import (run_lead_optimization, PERMISSIBLE_FLAVORS,
                         TASTE_FLAVORS, ODOUR_FLAVORS)

DATA       = Path.home() / "preprocessed" / "molplatte"
CHECKPOINT = Path.home() / "checkpoints" / "molplatte" / "molplatte-final-s911012.pt"
VOCAB      = DATA / "union_vocab" / "base-full__crossdocked__tastepocket" / "rgroup_vocab.pkl.gz"

opt = LeadOptimizer.load(CHECKPOINT, VOCAB, device="cuda")
opt.build_library(batch_size=1024)

print(f"library rows      {len(opt.vocab):,}")
print(f"effective size    {opt.vocab.effective_size():.0f}")
print(f"top-1 share       {100 * opt.vocab.frequency_prior[0]:.2f}%")
print(f"novel rows        {len(opt._novel):,}  (absent from the pretraining vocabulary)")

# The permissible flavour strings -- the 24 condvec bits the corpus stores.
print(f"\nflavours ({len(PERMISSIBLE_FLAVORS)}):")
print("  taste:", ", ".join(TASTE_FLAVORS))
print("  odour:", ", ".join(ODOUR_FLAVORS))
print("  other: odorless, unknown")

## Function Definitions for Lead Optimization

In [ ]:
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
from rdkit import Chem

from lead_optimization import LeadOptimizer
from lead_report import run_lead_optimization


def lead_optimization_pocketless(
    input_compound:   str | Chem.Mol,      # SMILES string or RDKit Mol
    lead_optimizer:   LeadOptimizer,
    flavor_condition: list[str],           # from PERMISSIBLE_FLAVORS
    top_k:            int = 10,
    max_decompositions: int = 4,
    gallery:          Optional[str | Path] = "lead_optimization.pdf",
):
    """Flavour-conditioned lead optimization, scored and rendered.

    Returns a LeadOptimizationReport:

        .table      deduplicated products, best score first, with
                    retrieval_score / MW / logP / QED / SAScore / NPScore,
                    plus d<prop> = product - input for each of those
        .reference  the INPUT compound's own specs, same five properties
        .compounds  the product SMILES, in table order
        .failures   suggestions that could not be assembled, WITH the reason
        .gallery    path to the PDF/PNG, or None if drawing was unavailable
        .results    the raw per-slot SlotResult list

    `retrieval_score` is the logQ-corrected value that actually ranks
    (sim/tau + log p(k)), not a similarity -- do not re-sort the table by
    anything else and expect the model's ordering back.

    Products are keyed on canonical SMILES across every decomposition and slot,
    so the same molecule proposed from three slots is ONE compound with
    n_slots=3, rather than three.

    `max_decompositions` is how many ways the input is cut into core + R-groups
    before any retrieval happens. It is the main control on how much comes back:
    the work is roughly max_decompositions x slots-per-decomposition x top_k, so
    raising it explores more of the molecule and costs proportionally. 1 uses
    only the highest-ranked decomposition.
    """
    return run_lead_optimization(
        input_compound, lead_optimizer, flavor_condition,
        top_k=top_k, max_decompositions=max_decompositions,
        pocket_condition=None, gallery=gallery,
    )


def lead_optimization_pocket(
    input_compound:   str | Chem.Mol,
    lead_optimizer:   LeadOptimizer,
    flavor_condition: list[str],
    pocket_condition: np.ndarray,          # 1280-d ESM-2 pocket embedding
    top_k:            int = 10,
    max_decompositions: int = 4,
    gallery:          Optional[str | Path] = "lead_optimization_pocket.pdf",
):
    """The same, additionally conditioned on a pocket.

    Get `pocket_condition` from a structure with

        pocket_embedding_from_structure(Path("receptor.cif"), device="cuda")

    READ THIS BEFORE TRUSTING THE OUTPUT. The shipped checkpoint's pocket
    reduction is zero-initialised and was never trained, so a pocket
    contributes EXACTLY ZERO. Pocket conditioning measured null at 0, 1,056 and
    168,096 trainable parameters, and the cause is a granularity mismatch --
    the pocket resolves chemotype, R-group retrieval needs exact fragments.
    See docs/step3_pocket_capacity_2026-09-09.md.

    So this does not take the pocket on faith. It runs retrieval twice, with
    and without, and sets `.pocket_changed_ranking`. When the pocket changed
    nothing it raises a RuntimeWarning and stamps the gallery, because an
    argument that silently does nothing is worse than one that says so. The
    signature and plumbing are ready for a trained pocket path; the checkpoint
    is not.
    """
    return run_lead_optimization(
        input_compound, lead_optimizer, flavor_condition,
        top_k=top_k, max_decompositions=max_decompositions,
        pocket_condition=pocket_condition, gallery=gallery,
    )


## Run


In [ ]:
VANILLIN = "COc1cc(C=O)ccc1O"

report = lead_optimization_pocketless(
    VANILLIN, opt, ["sweet", "woody"], top_k=8,
    max_decompositions=4,
    gallery="vanillin_sweet_woody.pdf",
)
print(report)
print(f"gallery: {report.gallery}")
if len(report.failures):
    print(f"{len(report.failures)} suggestion(s) could not be assembled")

# the starting compound, on the same scale as everything below
print("\nINPUT", report.input_smiles)
for k, v in report.reference.items():
    print(f"   {k:<8} {v:.3f}" if v is not None else f"   {k:<8} n/a")

# d<prop> is the change against that input -- an absolute QED means little,
# +0.10 on the starting compound means something.
report.table[["product", "rgroup", "retrieval_score",
              "MW", "dMW", "logP", "dlogP", "QED", "dQED",
              "SAScore", "dSAScore", "NPScore", "dNPScore",
              "is_novel", "n_slots", "is_input"]]